# KALE approximate replication on ArtCap (E4), ModelScope mPLUG start

KALE's official starting checkpoint (`mplug_large_v2.pth`) can't be downloaded (the host returns HTTP 403).
This notebook starts KALE's ArtCap training from **ModelScope's `iic/mplug_backbone_large_en`** instead: the pretrained English mPLUG-large backbone (not fine-tuned on any captioning data). It is the closest available match to the official checkpoint, but we can't confirm it is the identical file.
**Report the results as an approximate replication** and say why.

Run top to bottom with `SMOKE_TEST = True` first. When step 8 prints a table with a row, set it to `False` and re-run from step 5.

Settings

In [37]:
SMOKE_TEST = False      # True: tiny 64-image run to check everything works. False: full run.
FULL_EPOCHS = 60       # epochs in KALE's config
DRIVE_DIR = "/content/drive/MyDrive/KALE_files"
MS_MODEL = "iic/mplug_backbone_large_en"   # pretrained backbone; closest public match to mplug_large_v2.pth
# Alternative (already fine-tuned on COCO captions, further from the paper): "iic/mplug_image-captioning_coco_large_en"

Mount Drive, clone KALE, check GPU

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, glob, json, shutil
os.makedirs(DRIVE_DIR, exist_ok=True)
if not os.path.exists('/content/KALE'):
    !git clone https://github.com/Yanbei-Jiang/Artwork-Interpretation.git /content/KALE
%cd /content/KALE
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu)
if 'L4' in gpu:
    print('WARNING: L4 may not work with PyTorch 1.11. If the GPU test at the end of step 3 fails, switch to A100 or T4.')

Mounted at /content/drive
Cloning into '/content/KALE'...
remote: Enumerating objects: 685, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 685 (delta 78), reused 136 (delta 54), pack-reused 512 (from 1)
Receiving objects: 100% (685/685), 151.78 MiB | 16.42 MiB/s, done.
Resolving deltas: 100% (162/162), done.
/content/KALE
GPU: NVIDIA A100-SXM4-80GB, 81920 MiB


Python 3.9 environment for KALE 

`language_evaluation` comes from GitHub, because KALE's tar link is also 403.

In [3]:
PY = '/content/kale-env/bin/python'
!pip install -q uv
if not os.path.exists(PY):
    !uv venv /content/kale-env --python 3.9.16
!uv pip install -q --python {PY} torch==1.11.0+cu113 torchvision==0.12.0+cu113 --extra-index-url https://download.pytorch.org/whl/cu113
!uv pip install -q --python {PY} numpy==1.23.5 "transformers==4.18.0" pyyaml ruamel.yaml wandb oss2 ftfy regex tqdm six more-itertools "opencv-python-headless<4.9" timm==0.4.12 gdown
!uv pip install -q --python {PY} git+https://github.com/bckim92/language-evaluation.git
!apt-get install -y -qq default-jre-headless > /dev/null
!{PY} -c "import language_evaluation; language_evaluation.download('coco'); print('language_evaluation ok')"
!{PY} -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0)); print(torch.ones(2).cuda()*2)"

Using CPython 3.9.16
Creating virtual environment at: /content/kale-env
Activate with: source /content/kale-env/bin/activate
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/content/kale-env/lib/python3.9/site-packages/language_evaluation/__init__.py", line 16, in <module>
    from language_evaluation.coco_caption_py3.pycocotools.coco import COCO
  File "/content/kale-env/lib/python3.9/site-packages/language_evaluation/coco_caption_py3/pycocotools/coco.py", line 48, in <module>
    import matplotlib.pyplot as plt
  File "/content/kale-env/lib/python3.9/site-packages/matplotlib/__init__.py", line 1273, in <module>
    rcParams['backend'] = os.environ.get('MPLBACKEND')
  File "/content/kale-env/lib/python3.9/site-packages/matplotlib/__init__.py", line 741, in __setitem__
    raise ValueError(f"Key {key}: {ve}") from None
ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg',

Download the ModelScope mPLUG checkpoint

In [4]:
!pip install -q modelscope
from modelscope import snapshot_download
ms_dir = snapshot_download(MS_MODEL, cache_dir=DRIVE_DIR + '/modelscope')
print(ms_dir)
!ls -lhR {ms_dir} | head -40

2026-11-20 03:16:22,788 | INFO    | modelscope_hub.download | Downloading 12 files from iic/mplug_backbone_large_en@master


Downloading:   0%|          | 0/12 [00:00<?, ?file/s]

pytorch_model.bin:   6%|5         | 336M/5.66G [00:00<?, ?B/s]

/content/drive/MyDrive/KALE_files/modelscope/models/iic--mplug_backbone_large_en/snapshots/master
/content/drive/MyDrive/KALE_files/modelscope/models/iic--mplug_backbone_large_en/snapshots/master:
total 5.3G
-rw------- 1 root root  644 Nov 20 03:12 config_bert.json
-rw------- 1 root root 1.3K Nov 20 03:12 configuration.json
-rw------- 1 root root  606 Nov 20 03:12 config.yaml
-rw------- 1 root root 5.3G Nov 20 03:23 pytorch_model.bin
-rw------- 1 root root 4.5K Nov 20 03:12 README.md
drwx------ 2 root root 4.0K Nov 20 03:12 resources
-rw------- 1 root root 227K Nov 20 03:12 vocab.txt

/content/drive/MyDrive/KALE_files/modelscope/models/iic--mplug_backbone_large_en/snapshots/master/resources:
total 1.2M
-rw------- 1 root root 214K Nov 20 03:12 image-text-retrieval.jpg
-rw------- 1 root root 212K Nov 20 03:12 model.png
-rw------- 1 root root 366K Nov 20 03:12 retrieval_exp.png
-rw------- 1 root root 378K Nov 20 03:12 vqa_exp.png


Convert it to the format KALE loads

KALE's loader expects a file holding `{'model': state_dict}`. This also strips a leading `model.`/`module.` from parameter names if ModelScope added one.

In [27]:
import torch
# Keep the converted checkpoint on local disk: a multi-GB file written to Drive can be left
# half-uploaded if the runtime disconnects, which corrupts it. Rebuilding it takes about a minute.
CKPT = '/content/' + MS_MODEL.split('/')[-1] + '_kale.pth'   # separate file per source model

def ckpt_ok(path):
    try:
        torch.load(path, map_location='cpu')
        return True
    except Exception as e:
        print('converted checkpoint unreadable, rebuilding:', repr(e)[:120])
        os.remove(path)
        return False

if not (os.path.exists(CKPT) and ckpt_ok(CKPT)):
    weights = [p for p in glob.glob(ms_dir + '/**/*', recursive=True)
               if p.endswith(('.bin', '.pth', '.pt', '.safetensors')) and 'ViT' not in os.path.basename(p)]
    print('weight files:', weights)
    f = max(weights, key=os.path.getsize)
    print('source weights:', f, round(os.path.getsize(f) / 1e9, 2), 'GB (expect about 5.66 for the backbone)')
    if f.endswith('.safetensors'):
        from safetensors.torch import load_file
        sd = load_file(f)
    else:
        sd = torch.load(f, map_location='cpu')
    if isinstance(sd, dict) and ('model' in sd or 'module' in sd):
        sd = sd.get('model', sd.get('module'))
    for prefix in ('module.', 'model.'):
        if all(k.startswith(prefix) for k in sd):
            sd = {k[len(prefix):]: v for k, v in sd.items()}
    torch.save({'model': sd}, CKPT)
sd = torch.load(CKPT, map_location='cpu')['model']

# KALE's loader renames every key containing 'fusion' with key.replace('fusion.', ''), then deletes
# the old key. The official checkpoint names these 'fusion_encoder.fusion.encoder...', so they survive.
# ModelScope's names lack the inner 'fusion.', so the rename is a no-op and KALE deletes them.
# Add the inner 'fusion.' back so KALE's rename restores the right names.
fix = [k for k in sd if k.startswith('fusion_encoder.') and not k.startswith('fusion_encoder.fusion.')]
if fix:
    sd = {('fusion_encoder.fusion.' + k[len('fusion_encoder.'):] if k in fix else k): v for k, v in sd.items()}
    torch.save({'model': sd}, CKPT)
    print(f'renamed {len(fix)} fusion_encoder keys for KALE\'s loader')
print(len(sd), 'tensors')
for k in list(sd)[:12] + ['...'] + list(sd)[-8:]:
    print(' ', k, tuple(sd[k].shape) if k in sd else '')

weight files: ['/content/drive/MyDrive/KALE_files/modelscope/models/iic--mplug_backbone_large_en/snapshots/master/pytorch_model.bin']
source weights: /content/drive/MyDrive/KALE_files/modelscope/models/iic--mplug_backbone_large_en/snapshots/master/pytorch_model.bin 5.66 GB (expect about 5.66 for the backbone)
renamed 156 fusion_encoder keys for KALE's loader
2094 tensors
  temp ()
  image_queue (256, 65536)
  text_queue (256, 65536)
  queue_ptr (1,)
  visual_encoder.positional_embedding (77, 768)
  visual_encoder.text_projection (768, 768)
  visual_encoder.logit_scale ()
  visual_encoder.visual.class_embedding (1024,)
  visual_encoder.visual.positional_embedding (257, 1024)
  visual_encoder.visual.proj (1024, 768)
  visual_encoder.visual.conv1.weight (1024, 3, 14, 14)
  visual_encoder.visual.ln_pre.weight (1024,)
  ... 
  fusion_encoder_m.encoder.layer.11.crossattention.output.LayerNorm.weight (768,)
  fusion_encoder_m.encoder.layer.11.crossattention.output.LayerNorm.bias (768,)
  fusi

Other files and data

In [6]:
%cd {DRIVE_DIR}
# ViT-L-14.tar: KALE builds its image encoder from this. Use the ModelScope copy if it ships one, else KALE's link.
ms_vit = glob.glob(ms_dir + '/**/ViT-L-14.tar', recursive=True)
if not os.path.exists('ViT-L-14.tar'):
    if ms_vit:
        shutil.copy(ms_vit[0], 'ViT-L-14.tar')
    else:
        !wget -nc https://alice-open.oss-cn-zhangjiakou.aliyuncs.com/mPLUG/ViT-L-14.tar
assert os.path.getsize('ViT-L-14.tar') > 500e6, 'ViT-L-14.tar is missing or incomplete'
if not os.path.exists('kale_shared/data.zip'):
    !{PY.replace('python','gdown')} --folder "https://drive.google.com/drive/folders/1m-STuGx1U7c8tQ_KkqOv39CGaFTAs6WY" -O kale_shared
if not os.path.exists('artcap_download'):
    !{PY.replace('python','gdown')} --folder "https://drive.google.com/drive/folders/1l8NK8mvpkG3UfrTuTNyBZciIjFquV2ia" -O artcap_download

%cd /content/KALE
!ln -sf {DRIVE_DIR}/ViT-L-14.tar .
!unzip -o -q $(find {DRIVE_DIR}/kale_shared -name data.zip | head -1) 'data/ArtCap/*' -d /content/KALE
if not os.path.exists('data/ArtCap/images'):
    !unzip -o -q $(find {DRIVE_DIR}/artcap_download -name images.zip | head -1) -d /content/KALE/data/ArtCap

/content/drive/MyDrive/KALE_files
--2026-11-20 03:25:53--  https://alice-open.oss-cn-zhangjiakou.aliyuncs.com/mPLUG/ViT-L-14.tar
Resolving alice-open.oss-cn-zhangjiakou.aliyuncs.com (alice-open.oss-cn-zhangjiakou.aliyuncs.com)... 47.92.17.125
Connecting to alice-open.oss-cn-zhangjiakou.aliyuncs.com (alice-open.oss-cn-zhangjiakou.aliyuncs.com)|47.92.17.125|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 932224568 (889M) [application/x-tar]
Saving to: ‘ViT-L-14.tar’

ViT-L-14.tar        100%[===================>] 889.04M  16.1MB/s    in 4m 25s  

2026-11-20 03:30:31 (3.35 MB/s) - ‘ViT-L-14.tar’ saved [932224568/932224568]

Retrieving folder contents
Retrieving folder 1nlpM6wzq9UnJVk-3kW9QKIW7ohgovd5I bert_base_uncased
Processing file 1MS60BYi3l2UPY0aP6B4V-iN5zYnFxlOe config.json
Processing file 113uJiG15sjPD2i3yxdUqKInU5aQOU5nm pytorch_model.bin
Processing file 1HCkajvQbDHgI4ox_Y7KPFRyVnjKBXzs4 tokenizer_config.json
Processing file 1TuHCd1_wsoF4bm_5KsMMAuwLaK7Wy

In [7]:
# Check the layout matches what KALE's dataset code expects
import json, glob
train_val = json.load(open('data/ArtCap/artcap_train_val.json'))
test = json.load(open('data/ArtCap/artcap_test.json'))
print('train_val:', len(train_val), ' test:', len(test), ' (expected 2524 and 1082)')
print('example entry:', test[0])
missing = [e[0] for e in train_val + test if not os.path.exists(os.path.join('data/ArtCap/images', e[0]))]
print('images found:', len(glob.glob('data/ArtCap/images/*')), ' missing:', len(missing), missing[:5])
assert not missing, 'Some images are not where KALE expects them (data/ArtCap/images/<filename>). Check the unzip step.'

train_val: 2524  test: 1082  (expected 2524 and 1082)
example entry: ['34256.jpg', ['a painter was copying a model sleeping with a book in her arms', 'a woman looking at a book in front of a mirror', 'a girl with a man reading at a drawing table', 'a painter is sitting in a chair painting', 'an artist is drawing a woman reading a book']]
images found: 3606  missing: 0 []


Configure the run


Shrinks the batch to fit the GPU and raises gradient accumulation so each update still covers 512 examples, like the original (256 x 2).

In [38]:
import yaml, shutil
cfg_path = 'configs/config.yaml'
if not os.path.exists(cfg_path + '.orig'):
    shutil.copy(cfg_path, cfg_path + '.orig')
cfg = yaml.load(open(cfg_path + '.orig'), Loader=yaml.Loader)

BATCH = 16 if 'A100' in gpu else (8 if 'L4' in gpu else 4)
ACCUM = max(1, 512 // BATCH)
cfg['batch_size_train'] = BATCH
cfg['batch_size_test'] = BATCH
cfg['schedular']['epochs'] = 1 if SMOKE_TEST else FULL_EPOCHS
yaml.dump(cfg, open(cfg_path, 'w'))

# Smoke test: swap in tiny annotation files (originals are backed up and restored for the full run)
ann = 'data/ArtCap/'
for name in ['artcap_train_val.json', 'artcap_test.json']:
    if not os.path.exists(ann + name + '.full'):
        shutil.copy(ann + name, ann + name + '.full')
    data = json.load(open(ann + name + '.full'))
    json.dump(data[:64] if SMOKE_TEST else data, open(ann + name, 'w'))

RUN = 'bb_smoke' if SMOKE_TEST else 'bb_full'
OUT = f'/content/kale_out/{RUN}'          # checkpoints are ~GBs each, so keep them on local disk
RESULTS_DRIVE = f'{DRIVE_DIR}/results_{RUN}'
os.makedirs(OUT, exist_ok=True); os.makedirs(RESULTS_DRIVE, exist_ok=True)
print(f'batch={BATCH} accum={ACCUM} epochs={cfg["schedular"]["epochs"]} out={OUT}')

batch=16 accum=32 epochs=60 out=/content/kale_out/bb_full



Every 2 minutes, copies the caption results and log to Drive, and deletes all but the newest checkpoint.

In [35]:
import threading, time
def housekeeping():
    while True:
        try:
            ckpts = sorted(glob.glob(f'{OUT}/checkpoint_*.pth'))
            for c in ckpts[:-1]:
                os.remove(c)
            for f in glob.glob(f'{OUT}/result/result_epoch*.json') + glob.glob(f'{OUT}/log.txt'):
                if not f.endswith('_rank0.json'):
                    shutil.copy(f, RESULTS_DRIVE)
        except Exception as e:
            print('housekeeping:', e)
        time.sleep(120)
threading.Thread(target=housekeeping, daemon=True).start()

Train

In [39]:
%cd /content/KALE
# KALE scores captions after every epoch with its own evaluator, but never uses the result
# (only wandb.log). Its METEOR call crashes (BrokenPipeError), which kills training after epoch 0.
# Make that call non-fatal; step 8 scores the saved captions itself.
src = open('KALE_artcap.py').read()
if 'cal_metric skipped' not in src:
    old = 'result = cal_metric(result_file)'
    assert src.count(old) == 1, 'cal_metric call not found; paste this error'
    line = next(l for l in src.splitlines() if l.strip() == old)
    ind = line[:len(line) - len(line.lstrip())]
    new = (f"{ind}try:\n{ind}    result = cal_metric(result_file)\n"
           f"{ind}except Exception as e:\n{ind}    print('cal_metric skipped:', repr(e))")
    open('KALE_artcap.py', 'w').write(src.replace(line, new))
    print('patched KALE_artcap.py')
!MPLBACKEND=Agg WANDB_MODE=offline {PY} KALE_artcap.py --checkpoint {CKPT} --output_dir {OUT} --accum_steps {ACCUM} 2>&1 | tee {OUT}/train_output.txt
!cp {OUT}/train_output.txt {RESULTS_DRIVE}/
!ls {OUT} {OUT}/result

Streaming output truncated to the last 5000 lines.
Train Epoch: [30]  [145/789]  eta: 0:09:48  lr1: 0.000075  lr2: 0.000050  loss: 0.0060  time: 0.9084  data: 0.2629  max mem: 10720
Train Epoch: [30]  [150/789]  eta: 0:09:43  lr1: 0.000075  lr2: 0.000050  loss: 0.0065  time: 0.9027  data: 0.2605  max mem: 10720
Train Epoch: [30]  [155/789]  eta: 0:09:38  lr1: 0.000075  lr2: 0.000050  loss: 0.0079  time: 0.9017  data: 0.2590  max mem: 10720
Train Epoch: [30]  [160/789]  eta: 0:09:34  lr1: 0.000075  lr2: 0.000050  loss: 0.0065  time: 0.9058  data: 0.2605  max mem: 10720
Train Epoch: [30]  [165/789]  eta: 0:09:29  lr1: 0.000075  lr2: 0.000050  loss: 0.0092  time: 0.9074  data: 0.2627  max mem: 10720
Train Epoch: [30]  [170/789]  eta: 0:09:24  lr1: 0.000075  lr2: 0.000050  loss: 0.0097  time: 0.9115  data: 0.2661  max mem: 10720
Train Epoch: [30]  [175/789]  eta: 0:09:20  lr1: 0.000075  lr2: 0.000050  loss: 0.0077  time: 0.9183  data: 0.2731  max mem: 10720
Train Epoch: [30]  [180/789]  et

check data loaded properly 

In [30]:
log = open(f'{OUT}/train_output.txt').read()
import re
m = re.search(r'_IncompatibleKeys\(missing_keys=\[(.*?)\], unexpected_keys=\[(.*?)\]\)', log, re.S)
if m:
    missing = [k.strip(" '\"\n") for k in m.group(1).split(',') if k.strip()]
    unexpected = [k.strip(" '\"\n") for k in m.group(2).split(',') if k.strip()]
    def unused_fusion(k):
        hit = re.match(r'fusion_encoder\.encoder\.layer\.(\d+)\.', k)
        return k.startswith('fusion_encoder.embeddings.') or (hit and int(hit.group(1)) < 6)
    real_missing = [k for k in missing if not unused_fusion(k)]
    loaded_fusion = sorted({int(x) for x in re.findall(r'fusion_encoder\.(?:fusion\.)?encoder\.layer\.(\d+)\.', ' '.join(sd))})
    print('missing keys:', len(missing), '| expected unused fusion keys:', len(missing) - len(real_missing))
    print('fusion layers present in checkpoint:', loaded_fusion, '(should be [6, 7, 8, 9, 10, 11])')
    print('REAL missing keys:', len(real_missing), real_missing[:10])
    print('unexpected keys:', len(unexpected), '(pretraining-only parts; fine)')
    print('LOAD OK' if not real_missing and loaded_fusion == list(range(6, 12)) else 'LOAD PROBLEM: paste this output')
else:
    print('Load message not found. Searching the log for key messages:')
    for line in log.splitlines():
        if 'missing' in line.lower() or 'unexpected' in line.lower():
            print(line[:500])

missing keys: 162 | expected unused fusion keys: 162
fusion layers present in checkpoint: [6, 7, 8, 9, 10, 11] (should be [6, 7, 8, 9, 10, 11])
REAL missing keys: 0 []
unexpected keys: 905 (pretraining-only parts; fine)
LOAD OK


Reload saved captions from Drive

In [40]:
from google.colab import drive; drive.mount('/content/drive')
import os, glob, json, re
DRIVE_DIR = "/content/drive/MyDrive/KALE_files"
RESULTS_DRIVE = f"{DRIVE_DIR}/results_bb_full"
OUT = "/content/scored"                      # step 8 reads {OUT}/result/
os.makedirs(f"{OUT}/result", exist_ok=True)
!cp {RESULTS_DRIVE}/result_epoch*.json {OUT}/result/
os.makedirs('/content/KALE', exist_ok=True); os.chdir('/content/KALE')
!unzip -o -q $(find {DRIVE_DIR}/kale_shared -name data.zip | head -1) 'data/ArtCap/artcap_test.json' -d /content/KALE
print(sorted(int(re.findall(r'epoch(\d+)', f)[0]) for f in glob.glob(f'{OUT}/result/*.json'))[-3:])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[57, 58, 59]


score with metrics 

Same metrics as `helpers/metrics.py` (pycocoevalcap, PTB tokenizer), scaled to 0–100 like the KALE paper.

In [41]:
!pip install -q pycocoevalcap
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
import pandas as pd, re

refs = {e[0].split('/')[-1]: e[1] for e in json.load(open('data/ArtCap/artcap_test.json'))}

def score(path):
    preds = {str(r['image_id']): ' '.join(str(r['pred_caption']).split()) or 'none' for r in json.load(open(path))}
    ids = [i for i in refs if i in preds]
    tok = PTBTokenizer()
    gen = tok.tokenize({k: [{'caption': preds[i]}] for k, i in enumerate(ids)})
    exp = tok.tokenize({k: [{'caption': c} for c in refs[i]] for k, i in enumerate(ids)})
    out = {}
    b, _ = Bleu(4).compute_score(exp, gen, verbose=0)
    out.update({f'B{n}': v for n, v in enumerate(b, 1)})
    try:
        out['METEOR'], _ = float('nan')
    except Exception as e:
        print('METEOR failed, recorded as NaN:', repr(e)[:200])
        out['METEOR'] = float('nan')
    out['ROUGE_L'], _ = Rouge().compute_score(exp, gen)
    out['CIDEr'], _ = Cider().compute_score(exp, gen)
    out = {k: round(100 * v, 2) for k, v in out.items()}
    out['n'] = len(ids)
    out['mean_len'] = round(sum(len(preds[i].split()) for i in ids) / len(ids), 1)
    return out

files = sorted(glob.glob(f'{OUT}/result/result_epoch*.json'), key=lambda f: int(re.findall(r'epoch(\d+)', f)[0]))
files = [f for f in files if not f.endswith('_rank0.json')]
table = pd.DataFrame({int(re.findall(r'epoch(\d+)', f)[0]): score(f) for f in files}).T
table.index.name = 'epoch'
sample = json.load(open(files[-1]))[:5]
print('Sample predictions from the last epoch:')
for r in sample: print(' ', r['image_id'], '->', repr(r['pred_caption']))
table.to_csv(f'{RESULTS_DRIVE}/kale_scores_by_epoch.csv')
display(table)
print(table.to_string())
# Compare the last epoch with the published KALE numbers
paper = {'CIDEr': 36.51, 'B1': 59.78, 'B2': 41.24, 'B3': 28.56, 'B4': 20.14, 'METEOR': 19.39, 'ROUGE_L': 42.22}
last = table.iloc[-1]
check = pd.DataFrame({'ours (last epoch)': [last[k] for k in paper], 'KALE paper': list(paper.values())}, index=list(paper))
check['difference'] = check['ours (last epoch)'] - check['KALE paper']
check.to_csv(f'{RESULTS_DRIVE}/kale_vs_paper.csv')
print(f'\nEpoch {table.index[-1]} vs the KALE paper:')
print(check.round(2).to_string())

METEOR failed, recorded as NaN: TypeError('cannot unpack non-iterable float object')
METEOR failed, recorded as NaN: TypeError('cannot unpack non-iterable float object')
METEOR failed, recorded as NaN: TypeError('cannot unpack non-iterable float object')
METEOR failed, recorded as NaN: TypeError('cannot unpack non-iterable float object')
METEOR failed, recorded as NaN: TypeError('cannot unpack non-iterable float object')
METEOR failed, recorded as NaN: TypeError('cannot unpack non-iterable float object')
METEOR failed, recorded as NaN: TypeError('cannot unpack non-iterable float object')
METEOR failed, recorded as NaN: TypeError('cannot unpack non-iterable float object')
METEOR failed, recorded as NaN: TypeError('cannot unpack non-iterable float object')
METEOR failed, recorded as NaN: TypeError('cannot unpack non-iterable float object')
METEOR failed, recorded as NaN: TypeError('cannot unpack non-iterable float object')
METEOR failed, recorded as NaN: TypeError('cannot unpack non-iter

,B1,B2,B3,B4,METEOR,ROUGE_L,CIDEr,n,mean_len
epoch,,,,,,,,,
0,54.84,30.16,16.17,9.47,NaN,33.37,6.61,1082.0,9.4
1,52.60,28.60,14.92,8.23,NaN,32.77,6.17,1082.0,9.4
2,50.80,27.60,14.43,8.16,NaN,31.79,6.05,1082.0,9.4
3,50.81,27.60,14.53,8.40,NaN,32.40,6.00,1082.0,9.7
4,50.23,26.77,14.21,8.21,NaN,31.95,5.72,1082.0,9.7
5,49.79,26.61,13.97,7.85,NaN,31.11,6.01,1082.0,9.7
6,48.12,25.18,13.08,7.32,NaN,30.45,5.43,1082.0,9.8
7,46.60,24.18,12.26,6.70,NaN,30.07,4.88,1082.0,10.1
8,47.21,24.62,12.67,6.80,NaN,29.85,5.17,1082.0,10.1


          B1     B2     B3    B4  METEOR  ROUGE_L  CIDEr       n  mean_len
epoch                                                                     
0      54.84  30.16  16.17  9.47     NaN    33.37   6.61  1082.0       9.4
1      52.60  28.60  14.92  8.23     NaN    32.77   6.17  1082.0       9.4
2      50.80  27.60  14.43  8.16     NaN    31.79   6.05  1082.0       9.4
3      50.81  27.60  14.53  8.40     NaN    32.40   6.00  1082.0       9.7
4      50.23  26.77  14.21  8.21     NaN    31.95   5.72  1082.0       9.7
5      49.79  26.61  13.97  7.85     NaN    31.11   6.01  1082.0       9.7
6      48.12  25.18  13.08  7.32     NaN    30.45   5.43  1082.0       9.8
7      46.60  24.18  12.26  6.70     NaN    30.07   4.88  1082.0      10.1
8      47.21  24.62  12.67  6.80     NaN    29.85   5.17  1082.0      10.1
9      45.57  23.44  11.44  6.01     NaN    29.35   4.77  1082.0      10.3
10     45.63  23.44  11.56  6.09     NaN    29.21   4.47  1082.0      10.2
11     44.69  22.90  11.3

Diagnostics


In [42]:
import collections
print(table[['B1','B4','ROUGE_L','CIDEr','n','mean_len']].to_string())
best = table['CIDEr'].idxmax(); print('\nbest CIDEr epoch:', best, table.loc[best, 'CIDEr'])
preds = [r['pred_caption'] for r in json.load(open(files[-1]))]
print('\nunique captions in last epoch:', len(set(preds)), 'of', len(preds))
for c, k in collections.Counter(preds).most_common(8): print(k, repr(c))
print('\nfirst pred:', json.load(open(files[-1]))[0])
t = json.load(open('data/ArtCap/artcap_test.json'))[0]; print('first ref:', t[0], t[1][:2])

          B1    B4  ROUGE_L  CIDEr       n  mean_len
epoch                                               
0      54.84  9.47    33.37   6.61  1082.0       9.4
1      52.60  8.23    32.77   6.17  1082.0       9.4
2      50.80  8.16    31.79   6.05  1082.0       9.4
3      50.81  8.40    32.40   6.00  1082.0       9.7
4      50.23  8.21    31.95   5.72  1082.0       9.7
5      49.79  7.85    31.11   6.01  1082.0       9.7
6      48.12  7.32    30.45   5.43  1082.0       9.8
7      46.60  6.70    30.07   4.88  1082.0      10.1
8      47.21  6.80    29.85   5.17  1082.0      10.1
9      45.57  6.01    29.35   4.77  1082.0      10.3
10     45.63  6.09    29.21   4.47  1082.0      10.2
11     44.69  6.11    29.16   4.26  1082.0      10.6
12     45.11  6.00    29.15   4.68  1082.0      10.4
13     44.88  5.84    29.13   4.51  1082.0      10.4
14     43.94  5.70    28.69   4.62  1082.0      10.5
15     43.86  5.25    28.40   4.66  1082.0      10.3
16     43.99  5.83    28.44   4.47  1082.0    